# 02 — Screenshot batch

Drive the deployed ARCA in headless Chromium via Playwright and capture one PNG per (stem, view) pair. Outputs land under `outputs/screenshots/`. This is the interface to Jobdesk 4.

Each capture loads `BASE_URL/?model=<stem>&view=lit&yaw=<deg>&pitch=<deg>&shot=1` and waits for the viewer's `window.__arcaShotDone` sentinel before grabbing the download.

## 1. Setup

In [ ]:
!pip install --quiet playwright requests tqdm
!playwright install chromium

## 2. Config

Point `BASE_URL` at the deployed Hugging Face Space, or `http://localhost:5173` while testing locally with `npm run dev`. `VIEWS` is the camera matrix that Jobdesk 4 expects: front, three-quarter, and side.

In [ ]:
from pathlib import Path

BASE_URL = 'http://localhost:5173'
# BASE_URL = 'https://huggingface.co/spaces/lumicero/arca'

VIEWS = [
    {'name': 'front',         'yaw':  0.0, 'pitch':  0.0},
    {'name': 'three-quarter', 'yaw': 45.0, 'pitch': 20.0},
    {'name': 'side',          'yaw': 90.0, 'pitch':  0.0},
]

LIMIT = 3   # raise after a sanity-check pass; set to None for all
OUT_DIR = Path('..') / 'outputs' / 'screenshots'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PER_CAPTURE_TIMEOUT_MS = 15000

print('out:', OUT_DIR.resolve())

## 3. Load manifest

Pulled from the same URL the viewer reads, so the stem list always reflects what the deployed app actually serves.

In [ ]:
import requests

manifest_url = f'{BASE_URL.rstrip("/")}/manifest.json'
manifest = requests.get(manifest_url, timeout=15).json()
stems = [entry['stem'] for entry in manifest]
if LIMIT is not None:
    stems = stems[:LIMIT]
print(f'{len(stems)} stems queued')

## 4. Capture loop

One Chromium context for the whole batch keeps overhead low. Each navigation triggers `?shot=1`, which calls `canvas.toBlob` 500 ms after the model load and sets `window.__arcaShotDone = true`. We wait for that flag, then for the `download` event.

In [ ]:
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeout
from tqdm import tqdm

results = []
failures = []

with sync_playwright() as p:
    browser = p.chromium.launch()
    context = browser.new_context(
        accept_downloads=True,
        viewport={'width': 1280, 'height': 800},
    )
    page = context.new_page()

    plan = [(stem, view) for stem in stems for view in VIEWS]
    for stem, view in tqdm(plan, desc='capture', unit='shot'):
        url = (
            f"{BASE_URL.rstrip('/')}/?model={stem}&view=lit"
            f"&yaw={view['yaw']}&pitch={view['pitch']}&shot=1"
        )
        out = OUT_DIR / f"{stem}_{view['name']}.png"
        try:
            with page.expect_download(timeout=PER_CAPTURE_TIMEOUT_MS) as dl_info:
                page.goto(url, wait_until='load', timeout=PER_CAPTURE_TIMEOUT_MS)
                page.wait_for_function(
                    'window.__arcaShotDone === true',
                    timeout=PER_CAPTURE_TIMEOUT_MS,
                )
            download = dl_info.value
            download.save_as(out)
            results.append((stem, view['name'], out.stat().st_size))
        except PlaywrightTimeout as ex:
            failures.append((stem, view['name'], 'timeout'))
        except Exception as ex:
            failures.append((stem, view['name'], repr(ex)))

    browser.close()

print(f'captured: {len(results)}  failed: {len(failures)}')
for stem, view, why in failures[:10]:
    print(' ', stem, view, '->', why)

## 5. Summary

In [ ]:
from collections import defaultdict

by_stem = defaultdict(lambda: {'views': 0, 'bytes': 0})
for stem, view_name, size in results:
    by_stem[stem]['views'] += 1
    by_stem[stem]['bytes'] += size

print(f"{'stem':<24} {'views':>6} {'bytes':>12}")
for stem in sorted(by_stem):
    row = by_stem[stem]
    print(f"{stem:<24} {row['views']:>6} {row['bytes']:>12}")

print(f"\nfailures: {len(failures)}")